# Week 10 — Fairness Audit (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Fatima scheduled a Ministry review. The question: is this model equally reliable across all four cities?

**Before any code — write your hypothesis:**

## My Hypothesis

**Which city will have the lowest F1?** [WRITE HERE]

**Which bias type do you expect to dominate?** Representation / Measurement / Aggregation

**What is an acceptable equity gap for this deployment context?** [WRITE HERE]

## Part 1 — Per-City Evaluation

> ⏸️ **Pause and Predict**
> Overall test F1 = 0.91. If Kano = 0.75 and the other three cities are equal, what is their average? (Solve: 0.75 + 3x = 4 x 0.90.) If you reported only overall F1, how much of the Kano gap would a stakeholder miss?

In [ ]:
results={}
for city in sorted(test_df["city"].unique()):
    sub=test_df[test_df["city"]==city]
    loader=DataLoader(SatelliteDataset(sub,"datasets/images",ev),64,False,num_workers=0)
    preds,labels=[],[]
    with torch.no_grad():
        for imgs,lbls,_ in loader:
            preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy()); labels.extend(lbls.numpy())
    f1=f1_score(labels,preds,average="weighted",zero_division=0)
    results[city]={"f1":f1,"preds":preds,"labels":labels,"n":len(sub)}
    print(f"  {city:<8}: F1={f1:.3f}  (n={len(sub):,})")

best=max(results,key=lambda c:results[c]["f1"]); worst=min(results,key=lambda c:results[c]["f1"])
gap=results[best]["f1"]-results[worst]["f1"]
print(f"\nEquity gap: {gap:.3f}")
print(f"{'Gap > 0.15: deployment not recommended' if gap>0.15 else 'Gap 0.10-0.15: conditional approval' if gap>0.10 else 'Gap < 0.10: acceptable'}")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
cities=sorted(results.keys()); f1s=[results[c]["f1"] for c in cities]
axes[0].barh(cities,f1s,color=[CITY_CLR[c] for c in cities],height=0.6)
axes[0].axvline(0.80,color="#C0392B",linewidth=2,linestyle="--",label="Deployment threshold (0.80)")
axes[0].axvline(np.mean(f1s),color="black",linewidth=1.5,linestyle=":",label=f"Mean ({np.mean(f1s):.2f})")
axes[0].set_title(f"Per-City F1 — Equity Gap={gap:.2f}",fontweight="bold",loc="left")
axes[0].set_xlabel("Weighted F1"); axes[0].legend(fontsize=9)
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
cities_sorted=sorted(results.items(),key=lambda x:x[1]["f1"])
axes[1].barh([c for c,_ in cities_sorted],[v["f1"] for _,v in cities_sorted],
              color=[CITY_CLR[c] for c,_ in cities_sorted],height=0.6)
best_f1=max(v["f1"] for v in results.values()); worst_f1=min(v["f1"] for v in results.values())
axes[1].annotate("",xy=(best_f1,0.8),xytext=(worst_f1,0.8),
                 arrowprops=dict(arrowstyle="<->",color="#C0392B",lw=2))
axes[1].text((best_f1+worst_f1)/2,0.85,f"DeltaF1={gap:.2f}",ha="center",color="#C0392B",fontweight="bold",fontsize=10)
axes[1].set_title("Equity Gap",fontweight="bold",loc="left")
axes[1].set_xlabel("Weighted F1")
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
plt.tight_layout(); plt.savefig("outputs/equity_gap.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 2 — Threshold Routing

> 🧠 **What Will This Output?**
> The routing function flags patches with confidence < 0.70 for human review. Predict: what percentage of Kano patches will be routed? Will routing be more common for patches the model gets WRONG vs patches it gets RIGHT?

In [ ]:
def evaluate_with_threshold(model,df,city,threshold=0.70):
    sub=df[(df["split"]=="test")&(df["city"]==city)]
    loader=DataLoader(SatelliteDataset(sub,"datasets/images",ev),64,False,num_workers=0)
    preds,labels,confs,routed=[],[],[],[]
    model.eval()
    with torch.no_grad():
        for imgs,lbls,_ in loader:
            probs=torch.softmax(model(imgs.to(DEVICE)),dim=1)
            conf,pred=probs.max(dim=1)
            for i in range(len(lbls)):
                c=conf[i].item()
                if c<threshold: routed.append(True)
                else:
                    routed.append(False)
                    preds.append(pred[i].item()); labels.append(lbls[i].item()); confs.append(c)
    auto_f1=f1_score(labels,preds,average="weighted",zero_division=0) if preds else 0
    return auto_f1,sum(routed)/len(routed),len(preds)

print(f"{'Threshold':>10} {'Auto F1':>9} {'Review Rate':>12} {'n_auto':>8}")
print("-"*44)
worst_city=min(results,key=lambda c:results[c]["f1"])
for t in [0.0,0.50,0.60,0.70,0.80]:
    f1,rr,n=evaluate_with_threshold(model,df,worst_city,t)
    print(f"  {t:>8.2f}  {f1:>9.3f}  {rr:>12.1%}  {n:>8}")

## Part 3 — Fairness Brief

In [ ]:
BRIEF=f"""
VisionAI Lagos Fairness Audit Report
Model: EfficientNet-B0 | Date: [Week 10]

SUMMARY
Overall test F1: [FILL IN]
City range: {min(v['f1'] for v in results.values()):.3f} to {max(v['f1'] for v in results.values()):.3f}
Equity gap: {gap:.3f}

BIAS FINDINGS
1. Representation bias: Kano spectral signature differs from coastal majority.
   Model has lower confidence on Kano patches across all classes.
2. Aggregation bias: single model deployed across cities with meaningfully
   different spectral statistics.

MITIGATION APPLIED
Threshold routing at 0.70: Kano patches below this confidence level
are flagged for human review. Route rate: [FILL IN after running Part 2]

RESIDUAL RISK
A {gap:.2f}-point equity gap remains. Target: DeltaF1 < 0.08 before removing routing flag.
Action: collect 500 additional Kano patches in next quarter.

DEPLOYMENT DECISION: [CONDITIONAL APPROVAL / NOT APPROVED] pending threshold routing.
"""
with open("outputs/fairness_audit.md","w") as f: f.write(BRIEF)
print(BRIEF)

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Reporting only overall F1 | Always report equity gap |
| Tuning threshold on test data | Tune on val set; report on test |
| Claiming no bias from small gap | State gap + deployment context |
| Not documenting threshold decision | Record in model card |

## Challenge Task

> Implement **calibration curves** (reliability diagrams) per city. A model is calibrated if 80% confidence means 80% accuracy. Is the model equally well-calibrated across all four cities? A model that is overconfident on Kano is a specific fairness failure that threshold routing alone does not fix.
>
> Reference: Guo et al. (2017) On Calibration of Modern Neural Networks. ICML 2017.

## Week 10 Complete

Next: `week11_gradcam_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 10, you can now:

- Apply a fairness taxonomy to a CV model — what are the bias sources?
- Compute per-city precision, recall, and F1 with disaggregated metrics
- Write a fairness brief suitable for a compliance team
- Propose and document a mitigation strategy with before/after metrics

📤 **GitHub:** Push week10_fairness_STARTER.ipynb and fairness_brief.md. Commit: "Week 10: Fairness audit complete"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
